In [ ]:
import os
import boto3
import ijson
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score, classification_report, f1_score
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

In [ ]:
def parsing_json_s3():
  ACCESS_KEY = os.getenv("ACCESS_KEY")
  SECRET_KEY = os.getenv("SECRET_KEY")

  s3 = boto3.client(
    "s3",
    endpoint_url="https://storage.yandexcloud.net",
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY
  )

  response = s3.get_object(Bucket="slovo-mediapipe", Key="slovo_mediapipe.json")
  parser = ijson.kvitems(response["Body"], "")

  df = pd.read_csv("data\\annotations.csv", sep="\t")
  attachment_ids = df["attachment_id"].tolist()
  labels = df["text"].tolist()
  id_label_dict = {item: labels[i] for i, item in enumerate(attachment_ids)}

  daktyl = ["А", "Б", "В", "Г", "Д", "Е",
             "Ё", "Ж", "З", "И", "Й", "К",
             "Л", "М", "Н", "О", "П", "Р",
             "С", "Т", "У", "Ф", "Х", "Ц",
             "Ч", "Ш", "Щ", "Ь", "Ы", "Ъ", "Э",
             "Ю", "Я"]

  keys, coords, items = [], [], []
  for key, value in parser:
    if id_label_dict[key] in daktyl:
      keys.append(key)
      for i in range(len(value)):
        items.append(value[i]["hand 1"])
      coords.append(items)
      items = []

  return keys, coords


In [ ]:
def features_extraction(coords: list) -> list:
  stat_coords = []
  for i in tqdm(range(len(coords))):
    try:
      video_np = np.array(coords[i], dtype=np.float32)

      video_mean = np.mean(video_np, axis=0)
      video_min = np.min(video_np, axis=0)
      video_max = np.max(video_np, axis=0)
      video_std = np.std(video_np, axis=0)
      video_var = np.var(video_np, axis=0)
      video_range = video_max-video_min
      video_median = np.median(video_np, axis=0)
      video_25 = np.percentile(video_np, 25, axis=0)
      video_75 = np.percentile(video_np, 75, axis=0)


      all_stat = np.array([video_mean,
                        video_min,
                        video_max,
                        video_std,
                        video_var,
                        video_range,
                        video_median,
                        video_25,
                        video_75])
      stat_coords.append(all_stat.flatten())
      all_stat = []
    except:
      print(i, " ", coords[i])
  return stat_coords

In [ ]:
def id_label():
  df = pd.read_csv("data\\annotations.csv", sep="\t")
  attachment_ids = df["attachment_id"].tolist()
  labels = df["text"].tolist()
  id_label_dict = {item: labels[i] for i, item in enumerate(attachment_ids)}
  return id_label_dict

In [ ]:
def labels_preprocessing(id_label_dict: dict, keys_corrected: list) -> list:
  labels_classification = [id_label_dict[id] for id in keys_corrected]
  encoder = LabelEncoder()
  labels_encoded = np.int32(encoder.fit_transform(labels_classification))
  return labels_encoded, encoder.classes_

In [ ]:
def coords_preprocessing(coords: list) -> list:
  scaler = StandardScaler()
  coords_standarded = scaler.fit_transform(coords)
  return coords_standarded

In [ ]:
def model_classification(model_name: str, X: list, y: list, target_names: list, labels: list) -> float:
  models = {
        "logistic_regression": LogisticRegression(multi_class="multinomial",
                                                  penalty="l2",
                                                  solver="saga"),
        "svm": SVC(C=1,
                   kernel="linear"),
        "knn": KNeighborsClassifier(n_neighbors=3,
                                    metric="euclidean",
                                    p=2),
        "naive_bayes": GaussianNB(var_smoothing=1e-12),
        "random_forest": RandomForestClassifier(bootstrap=True,
                                                max_depth=30,
                                                max_features="sqrt",
                                                min_samples_leaf=4,
                                                min_samples_split=2,
                                                n_estimators=100),
        "lightgbm": LGBMClassifier(objective="multiclass",
                                   n_estimators=100,
                                   verbose=-1),
        "xgboost": XGBClassifier(objective='multi:softprob',
                                 n_estimators=100,
                                 verbose=-1),
        "catboost": CatBoostClassifier(loss_function='MultiClass',
                                       n_estimators=10,
                                       verbose=0)
    }
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)
  model = models[model_name]
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)
  accuracy_metric, f1_metric = accuracy_score(y_pred, y_test), f1_score(y_pred, y_test, average="macro", zero_division=0)
  classification_report_metric = classification_report(y_pred, y_test, zero_division=0, target_names=target_names, labels=labels)

  return accuracy_metric, f1_metric, classification_report_metric

In [ ]:
def dense_model(input_dim, num_classes):
  inputs = layers.Input(shape=(input_dim,))
  x = layers.Dense(128, activation="relu")(inputs)
  x = layers.BatchNormalization()(x)
  x = layers.Dropout(0.3)(x)

  x = layers.Dense(64, activation="relu")(inputs)
  x = layers.BatchNormalization()(x)
  x = layers.Dropout(0.3)(x)

  outputs = layers.Dense(num_classes, activation="softmax")(x)
  model = models.Model(inputs, outputs)
  return model

In [ ]:
def dense_classification(X: list, y: list):
  folds = KFold(n_splits=5, random_state=42, shuffle=True)
  accuracy_all = []
  for i, (train_index, test_index) in enumerate(folds.split(X, y)):
    X_train, X_test = np.array(X)[train_index], np.array(X)[test_index]
    y_train, y_test = np.array(y)[train_index], np.array(y)[test_index]


    X_train_tf, X_test_tf = tf.convert_to_tensor(X_train, dtype=np.float32), tf.convert_to_tensor(X_test, dtype=np.float32)
    y_train_tf, y_test_tf = tf.convert_to_tensor(y_train, dtype=np.int32), tf.convert_to_tensor(y_test, dtype=np.int32)

    model = dense_model(567, 1000)
    model.compile(optimizer="Adam", metrics=["accuracy"], loss='sparse_categorical_crossentropy')

    model.fit(X_train_tf, y_train_tf, epochs=100, batch_size=32, validation_data=(X_test_tf, y_test_tf))
    _, accuracy = model.evaluate(X_test_tf, y_test_tf)
    accuracy_all.append(accuracy)
  return accuracy_all

In [ ]:
data = parsing_json_s3()
keys_corrected, coords = data[0], data[1]
stat_coords = features_extraction(coords)

In [ ]:
print(len(keys_corrected))
print(len(stat_coords))

In [ ]:
keys_all = []
for i in range(len(keys_corrected)):
  if i not in [3179, 10146, 10335, 10368, 10395, 10412, 10413, 11310, 17308, 18768]:
    keys_all.append(keys_corrected[i])

In [ ]:
print(len(stat_coords))
print(len(stat_coords[0]))
print(len(keys_all))

In [ ]:
coords_preprocessed = coords_preprocessing(stat_coords)

In [ ]:
print(len(coords_preprocessed))
print(len(coords_preprocessed[0]))

In [ ]:
id_label_dict = id_label()
output = labels_preprocessing(id_label_dict, keys_all)
labels_encoded = output[0]
print(len(labels_encoded))

In [ ]:
target_names = ["А", "Б", "В", "Г", "Д", "Е",
             "Ё", "Ж", "З", "И", "Й", "К",
             "Л", "М", "Н", "О", "П", "Р",
             "С", "Т", "У", "Ф", "Х", "Ц",
             "Ч", "Ш", "Щ", "Ь", "Ы", "Ъ", "Э",
             "Ю", "Я"]

In [ ]:
model_name = ""
metrics = model_classification(model_name, coords_preprocessed, labels_encoded, target_names, target_names)
print("ACCURACY ", metrics[0])

In [ ]:
data_dense = dense_classification(X=coords_preprocessed, y=labels_encoded)
print("ACCURACY ALL: ", data_dense)
print("ACCURACY MEAN: ", np.mean(data_dense))